In [2]:
!pip3 install -r requirements.txt

  Cloning https://github.com/evfro/polara.git to /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-7a_nwzeh/polara_9e6def30db1d4d148af594fc94ce1c8c
  Running command git clone --filter=blob:none --quiet https://github.com/evfro/polara.git /private/var/folders/0y/g49_sx310b90y_dbbtzchfpc0000gn/T/pip-install-7a_nwzeh/polara_9e6def30db1d4d148af594fc94ce1c8c
  Resolved https://github.com/evfro/polara.git to commit 86a5d247335242f31baf8fb68e472b651ae6770f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 684.4/684.4 kB 1.

In [3]:
from typing import Callable, Dict, List, Tuple, Any, Optional
from collections import defaultdict


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
from torch.utils.data import Dataset, DataLoader


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda, split_and_reindex)

/Users/timurmif/ACADEMY/daml/deep-recsys-hse/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [5]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history[-max_seq_len:])
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [6]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [7]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128

config = {
    "ml": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    # "yambda": {
    #     "max_seq_len": 200,
    #     "test_quantile": 0.1,
    #     "interactions_path": "yambda-10m.parquet", # из ноута Кирилла (мб кто подкинет ссылку)
    #     "col_mapping": {
    #         "uid": "user_id",
    #         "item_id": "item_id",
    #         "timestamp": "timestamp",
    #     },
    # },
    # "amzn-books": {
    #     "max_seq_len": 50,
    #     "test_quantile": 0.1,
    #     "interactions_path": "amzn-books-20m.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
    #     "col_mapping": {
    #         "user_id": "user_id",
    #         "parent_asin": "item_id",
    #         "rating": "feedback",
    #         "timestamp": "timestamp",
    #     },
    # },
}

In [8]:
ml_df = load_ml20m(config["ml"]["interactions_path"], config=config["ml"])
# yambda_df = load_yambda(config["yambda"]["interactions_path"], config=config["yambda"])
# amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [10]:
ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml"])
# yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
# amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [11]:
datasets = {
    "ml": {
        "train": ml_train,
        "test": ml_test,
        "desc": ml_data_description,
    },
    # "yambda": {
    #     "train": yambda_train,
    #     "test": yambda_test,
    #     "desc": yambda_data_description,
    # },
    # "amzn-books": {
    #     "train": amzn_train,
    #     "test": amzn_test,
    #     "desc": amzn_data_description,
    # },
}

In [12]:
def build_histories(df: pd.DataFrame, desc: Dict) -> Dict:
    return df.sort_values([desc['users'], desc['timestamp']], ascending=True).groupby(desc['users'])[desc['items']].apply(list).to_dict()

for name, d in datasets.items():
    train, test, desc = d['train'], d['test'], d['desc']
    max_seq_len = config[name]['max_seq_len']

    histories = build_histories(train, desc)
    targets = build_histories(test, desc)
    
    d['histories'] = histories
    d['targets'] = targets  

    # d['train_dataset'] = TransfromerTrainDataset(histories=histories, max_seq_len=max_seq_len)
    # d['eval_dataset'] = TransfromerEvalDataset(histories=histories, targets=targets, max_seq_len=max_seq_len)

    # d['train_dataloader'] = DataLoader(
    #     dataset=d['train_dataset'],
    #     batch_size=TRAIN_BATCH_SIZE,
    #     shuffle=True,
    #     collate_fn=collate_fn,
    #     drop_last=True
    # )
    # d['eval_dataloader'] = DataLoader(
    #     dataset=d['eval_dataset'],
    #     batch_size=EVAL_BATCH_SIZE,
    #     shuffle=False,
    #     collate_fn=collate_fn,
    #     drop_last=False
    # )

In [13]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,ml,18000184,353348,18353532,0.981,0.019,125040,4958,17951


In [ ]:
class RandomRecommender:
    def __init__(self, n_items: int):
        self.n_items = n_items

    def fit(self, train_histories: Dict[int, List[int]]):
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        recs = np.argsort(np.random.rand(len(user_ids), self.n_items), axis=1)[:, :topk]
        return dict(zip(user_ids, recs.tolist()))


class TopPopRecommender:
    def __init__(self, n_items: int):
        self.n_items = n_items
        self.popular_items = None

    def fit(self, train_histories: Dict[int, List[int]]):
        all_items = np.concatenate(list(train_histories.values()))
        counts = np.bincount(all_items, minlength=self.n_items)
        self.popular_items = np.argsort(counts)[::-1].tolist()
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        top = self.popular_items[:topk]
        return dict(zip(user_ids, [top] * len(user_ids)))

In [ ]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    candidates = np.asarray(candidates[:topk])
    targets = np.asarray(targets)
    gu_size = len(np.unique(targets))

    hits = np.isin(candidates, targets).astype(np.int32)

    hr = float(hits.sum() > 0)
    recall = hits.sum() / min(topk, gu_size)

    weights = 1 / np.log2(np.arange(2, topk + 2))
    ndcg = (hits[:topk] * weights[:topk]).sum() / weights[:min(topk, gu_size)].sum()

    return {'hitrate': hr,
            'recall': recall,
            'ndcg': ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:

    metrics_sum = [0, 0, 0]
    items = set()

    for uid, targets in targets.items():
        items.update(candidates[uid])

        um = get_metrics(targets, candidates[uid], topk)
        for k, v in enumerate(um.values()):
            metrics_sum[k] += v

    metrics = defaultdict(float,
     {'hitrate': metrics_sum[0] / len(candidates),
      'recall': metrics_sum[1] / len(candidates),
      'ndcg': metrics_sum[2] / len(candidates),
      'coverage': len(items) / catalog_size})

    return metrics

NameError: name 'List' is not defined

In [ ]:
# del ml_df, yambda_df, amzn_df
# for d in datasets.values():
#     del d['train'], d['test']

In [ ]:
# results = []

# for name, d in datasets.items():
#     histories = d['histories']
#     targets = d['targets']
#     desc = d['desc']

#     random_model = RandomRecommender(n_items=desc['n_items']).fit(histories)
#     toppop_model = TopPopRecommender(n_items=desc['n_items']).fit(histories)

#     user_ids = list(targets.keys())

#     for model_name, recs in [('random', random_model.predict(user_ids, topk=100)), ('toppop', toppop_model.predict(user_ids, topk=100))]:
#         metrics = evaluate(targets, recs, catalog_size=desc['n_items'], topk=100)
#         results.append({'dataset': name, 'model': model_name, **metrics})

# pd.DataFrame(results)